In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import imageio
from contextlib import nullcontext
from types import SimpleNamespace
import torch
import torchvision.transforms as T
from accelerate import Accelerator
from accelerate import PartialState
from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection
import lpips
import sys
from pathlib import Path
import matplotlib.pyplot as plt 



project_root = Path().resolve().parent
sys.path.append(str(project_root))


from material_aware_flare.eval import calculate_metrics

In [2]:
import sys
sys.path.append("../external/diffusion-renderer") 

from src.pipelines.pipeline_rgbx import RGBXVideoDiffusionPipeline
from utils.utils_rgbx import convert_rgba_to_rgb_pil
from utils.utils_rgbx_inference import touch, find_images_recursive, base_plus_ext, \
    group_images_into_videos, split_list_with_overlap, resize_upscale_without_padding

In [3]:
if not Path("./checkpoints/diffusion_renderer-inverse-svd").exists():
    !python ../external/diffusion-renderer/utils/download_weights.py --repo_id nexuslrf/diffusion_renderer-inverse-svd

In [3]:
eval_transform_lpips = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

cfg = SimpleNamespace(
    inference_model_weights="/home/hleonhard/adl4cv_ws25-26_Relightable-Avatars/notebooks/checkpoints/diffusion_renderer-inverse-svd",
    inference_input_dir="/home/hleonhard/data/flare_subject_data/001/MVI_1812/image/",
    inference_save_dir="/home/hleonhard/data//output_diffusionrenderer_001/MVI_1812//",

    # Inference Parameters
    inference_n_frames=3,
    overlap_n_frames=2,
    inference_n_steps=20,
    chunk_mode='all',  # 'first' or 'all'
    model_passes=[ 'roughness', 'normal', 'diffuse_albedo'],
    inference_res=[512, 512],

    # Model Config 
    weight_dtype='fp16',
    cond_mode="skip",
    use_deterministic_mode=False,
    seed=42,
    autocast=True,

    # SVD-specific parameters
    inference_min_guidance_scale=1.0,
    inference_max_guidance_scale=3.0,
    fps=7,
    motion_bucket_id=127,
    cond_aug=0,
    decode_chunk_size=None,

    # Data Loading
    image_group_mode="folder", # 'folder' or 'individual'
    subsample_every_n_frames=1,
    image_extensions=['.png', '.jpg', '.jpeg'],

    # Saving
    save_image=False,
    save_video=False,
    save_video_fps=7,

    # Evaluation
    do_evaluation=True,
    flare_albedo_dir="/home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/albedo/",
    flare_normal_dir="/home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/normal/",
    flare_roughness_dir="/home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/roughness/",
    flare_mask_dir='/home/hleonhard/data/flare_subject_data/001/MVI_1812/mask/',
)

# Post-process config (from original code)
cfg.inference_height, cfg.inference_width = cfg.inference_res
if cfg.weight_dtype == 'fp16':
    cfg.torch_dtype = torch.float16
elif cfg.weight_dtype == 'fp32':
    cfg.torch_dtype = torch.float32

assert cfg.flare_albedo_dir is not None, "flare_albedo_dir must be provided for evaluation"
assert cfg.flare_normal_dir is not None, "flare_normal_dir must be provided for evaluation"
assert cfg.flare_roughness_dir is not None, "flare_roughness_dir must be provided for evaluation"
assert cfg.flare_mask_dir is not None
assert os.path.isdir(cfg.flare_albedo_dir), f"Flare albedo dir not found: {cfg.flare_albedo_dir}"
assert os.path.isdir(cfg.flare_normal_dir), f"Flare normal dir not found: {cfg.flare_normal_dir}"
assert os.path.isdir(cfg.flare_roughness_dir), f"Flare roughness dir not found: {cfg.flare_roughness_dir}"
assert os.path.isdir(cfg.flare_mask_dir)
print("Evaluation enabled. Comparing against:")
print(f"  Albedo: {cfg.flare_albedo_dir}")
print(f"  Normal: {cfg.flare_normal_dir}")
print(f"  Roughness: {cfg.flare_roughness_dir}")


Evaluation enabled. Comparing against:
  Albedo: /home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/albedo/
  Normal: /home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/normal/
  Roughness: /home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/roughness/


In [5]:
torch.cuda.empty_cache()
import gc
gc.collect()

60

In [6]:
# Setup Accelerator
accelerator = Accelerator()
distributed_state = PartialState()
device = accelerator.device

print(f"Using device: {device}")

missing_kwargs = {}
missing_kwargs["cond_mode"] = cfg.cond_mode
missing_kwargs["use_deterministic_mode"] = cfg.use_deterministic_mode

if os.path.exists(cfg.inference_model_weights):
    model_weights_subfolders = os.listdir(cfg.inference_model_weights)
else:
    model_weights_subfolders = []

if "image_encoder" not in model_weights_subfolders:
    print("Downloading missing image_encoder from StabilityAI...")
    missing_kwargs["image_encoder"] = CLIPVisionModelWithProjection.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid", subfolder="image_encoder",
    )
    assert cfg.cond_mode != "image", "Image encoder missing but cond_mode is 'image'"
if "feature_extractor" not in model_weights_subfolders:
    print("Downloading missing feature_extractor from StabilityAI...")
    missing_kwargs["feature_extractor"] = CLIPImageProcessor.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid", subfolder="feature_extractor",
    )
    assert cfg.cond_mode != "image", "Feature extractor missing but cond_mode is 'image'"

pipeline = RGBXVideoDiffusionPipeline.from_pretrained(cfg.inference_model_weights, **missing_kwargs)
pipeline = pipeline.to(device)
pipeline = pipeline.to(cfg.torch_dtype)
pipeline.set_progress_bar_config(disable=True)

lpips_model = lpips.LPIPS(net='vgg').to(device).to(cfg.torch_dtype)

Using device: cuda


cannot get type annotation for Parameter env_encoder of <class 'src.pipelines.pipeline_rgbx.RGBXVideoDiffusionPipeline'>.


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


/home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.

In [4]:
# Diffusion Renderer requires the images to be grouped into sequences
validation_image_paths = find_images_recursive(
    cfg.inference_input_dir, image_extensions=cfg.image_extensions
)
print(f"Found {len(validation_image_paths)} total images.")

validation_video_list = group_images_into_videos(
    validation_image_paths,
    image_group_mode=cfg.image_group_mode,
    subsample_every_n_frames=cfg.subsample_every_n_frames,
)
print(f"Grouped into {len(validation_video_list)} videos/sequences.")

os.makedirs(cfg.inference_save_dir, exist_ok=True)
success_signal_dir = os.path.join(cfg.inference_save_dir, "TMP_SUCCESS_SIGNAL")
os.makedirs(success_signal_dir, exist_ok=True)

if os.path.exists(success_signal_dir):
    filtered_validation_video_list = []
    for input_image_relative_path_list in validation_video_list:
        video_relative_base_name = base_plus_ext(
            input_image_relative_path_list[0], mode=cfg.image_group_mode
        )[0]
        input_image_relative_path_chunks = split_list_with_overlap(
            input_image_relative_path_list,
            cfg.inference_n_frames,
            cfg.overlap_n_frames,
            chunk_mode=cfg.chunk_mode,
        )
        if not input_image_relative_path_chunks:
            continue

        max_chunk_ind = len(input_image_relative_path_chunks) - 1
        success_signal_str = (
            video_relative_base_name.replace("/", "--") + f".{max_chunk_ind:04d}"
        )
        success_signal_path = os.path.join(success_signal_dir, success_signal_str)

        if os.path.exists(success_signal_path):
            print(f"Skipping already processed video: {success_signal_str}")
        else:
            filtered_validation_video_list.append(input_image_relative_path_list)

    validation_video_list = filtered_validation_video_list
    print(f"{len(validation_video_list)} videos remaining to process.")

processing_list = validation_video_list

Found 1825 total images.
Grouped into 1 videos/sequences.
1 videos remaining to process.


In [ ]:
# grap the names of the first 500 chunks
chunks = []

for i, input_image_relative_path_list in tqdm(
    enumerate(processing_list), desc="Processing Videos"
):
    video_relative_base_name = base_plus_ext(
        input_image_relative_path_list[0], mode=cfg.image_group_mode
    )[0]

    # Split into chunks
    input_image_relative_path_chunks = split_list_with_overlap(
        input_image_relative_path_list,
        cfg.inference_n_frames,
        cfg.overlap_n_frames,
        chunk_mode=cfg.chunk_mode,
    )

    for chunk_ind in tqdm(
        range(len(input_image_relative_path_chunks)),
        desc="  Processing Chunks",
        leave=False,
    ):
        current_image_relative_path_list = input_image_relative_path_chunks[chunk_ind]
        chunks.append(current_image_relative_path_list[0])
        current_image_relative_path_list[0]

Processing Videos: 0it [00:00, ?it/s]

  Processing Chunks:   0%|          | 0/1823 [00:00<?, ?it/s]

In [18]:
import pandas as pd

n = len(chunks)
subset = pd.Series(chunks[:n//3])
subset.to_csv("diffusionrenderer_001_mvi_1812_subset.csv")

In [46]:
chunks[0]

'1.png'

In [30]:
marker = pd.read_csv("diffusionrenderer_001_mvi_1812_subset.csv", index_col=0)

In [39]:
'1.png' in marker.iloc[:,0].values

True

In [47]:
marker.iloc[:,0].values[0]

'1.png'

In [ ]:
from glob import glob
images = glob('/home/hleonhard/data/flare_subject_data/001/MVI_1814/image/**.png')
masks = glob('/home/hleonhard/data/flare_subject_data/001/MVI_1814/mask/**.png')
marker  = pd.read_csv("diffusionrenderer_001_mvi_1812_subset.csv", index_col=0)
images = [image for image in images if image.split('/')[-1] in marker.iloc[:,0].values]
masks = [mask for mask in masks if mask.split('/')[-1] in marker.iloc[:,0].values]

607

In [26]:
pd.read_csv("diffusionrenderer_001_mvi_1812_subset.csv", index_col=0)

,0
0,1.png
1,10.png
2,100.png
3,1000.png
4,1001.png
...,...
602,1540.png
603,1541.png
604,1542.png
605,1543.png


In [ ]:
# run with screen in a stand alone file :)

all_metrics_data = {'diffuse_albedo': [], 'normal': [], 'roughness': []}

if 'lpips_model' in locals() or 'lpips_model' in globals():
    lpips_model = lpips_model.to(device, dtype=torch.float32)
    lpips_model.eval()

print("Starting inference and evaluation...")
for i, input_image_relative_path_list in tqdm(
    enumerate(processing_list), desc="Processing Videos"
):
    video_relative_base_name = base_plus_ext(
        input_image_relative_path_list[0], mode=cfg.image_group_mode
    )[0]

    # Split into chunks
    input_image_relative_path_chunks = split_list_with_overlap(
        input_image_relative_path_list,
        cfg.inference_n_frames,
        cfg.overlap_n_frames,
        chunk_mode=cfg.chunk_mode,
    )
    if len(input_image_relative_path_chunks) == 0:
        continue

    if cfg.save_image:
        os.makedirs(
            os.path.join(cfg.inference_save_dir, f"{video_relative_base_name}"),
            exist_ok=True,
        )

    for chunk_ind in tqdm(
        range(len(input_image_relative_path_chunks)),
        desc="  Processing Chunks",
        leave=False,
    ):
        success_signal_str = (
            video_relative_base_name.replace("/", "--") + f".{chunk_ind:04d}"
        )
        success_signal_path = os.path.join(success_signal_dir, success_signal_str)
        if os.path.exists(success_signal_path):
            print(f"Skipping chunk: {success_signal_str}")
            continue

        current_image_relative_path_list = input_image_relative_path_chunks[chunk_ind]
        # check if we skip this index
        if current_image_relative_path_list[0] in marker.iloc[:,0].values:
            # already saw this 
            print(f"Skipping chunk: {current_image_relative_path_list[0]}")
            continue
        # Fill frames to inference_n_frames
        while len(current_image_relative_path_list) < cfg.inference_n_frames:
            current_image_relative_path_list.append(
                current_image_relative_path_list[-1]
            )

        # Process input image
        input_images_uint8 = []
        for ind in range(cfg.inference_n_frames):
            input_path = os.path.join(
                cfg.inference_input_dir, current_image_relative_path_list[ind]
            )
            input_image_pil = Image.open(input_path)
            input_image_pil = convert_rgba_to_rgb_pil(
                input_image_pil, background_color=(0, 0, 0)
            )

            if ind == 0:
                width, height = input_image_pil.size
                if width != cfg.inference_width or height != cfg.inference_height:
                    input_image_pil = resize_upscale_without_padding(
                        input_image_pil, cfg.inference_height, cfg.inference_width
                    )
                    width, height = input_image_pil.size
            else:
                if (
                    width != input_image_pil.size[0]
                    or height != input_image_pil.size[1]
                ):
                    input_image_pil = input_image_pil.resize(
                        (width, height), resample=Image.BILINEAR
                    )

            if cfg.save_image:
                save_path = os.path.join(
                    cfg.inference_save_dir,
                    f"{video_relative_base_name}/{chunk_ind:04d}.{ind:04d}.rgb.png",
                )
                input_image_pil.save(save_path)

            input_images_uint8.append(np.asarray(input_image_pil))

        # Formatting input
        input_images = (
            np.stack(input_images_uint8, axis=0)[None, ...].astype(np.float32) / 255.0
        )  # (1, F, H, W, C)
        cond_images = {"rgb": input_images}
        cond_labels = {"rgb": "vae"}
        if cfg.cond_mode == "image":
            cond_images["clip_img"] = input_images[
                :, 0:1, ...
            ]  # NOTE: clip uses first frame only
            cond_labels["clip_img"] = "clip"

        viz_images_uint8 = input_images_uint8
        for inference_pass in cfg.model_passes:
            cond_images["input_context"] = inference_pass

            # DiffusionRenderer Pipeline
            generator = None
            if cfg.seed is not None:
                generator = torch.Generator(device=device).manual_seed(cfg.seed)

            autocast_ctx = (
                torch.autocast(device.type, enabled=cfg.autocast)
                if not torch.backends.mps.is_available()
                else nullcontext()
            )

            with autocast_ctx:
                inference_image_list = pipeline(
                    cond_images,
                    cond_labels,
                    height=height,
                    width=width,
                    num_frames=cfg.inference_n_frames,
                    num_inference_steps=cfg.inference_n_steps,
                    min_guidance_scale=cfg.inference_min_guidance_scale,
                    max_guidance_scale=cfg.inference_max_guidance_scale,
                    fps=cfg.fps,
                    motion_bucket_id=cfg.motion_bucket_id,
                    noise_aug_strength=cfg.cond_aug,
                    generator=generator,
                    decode_chunk_size=cfg.decode_chunk_size,
                ).frames[0]

            # RGBX Pipeline

            # Save images and run evaluation
            for ind in range(len(inference_image_list)):
                if cfg.save_image or True:
                    # save images for jonathan
                    save_path = os.path.join(
                        cfg.inference_save_dir,
                        f"{video_relative_base_name}/{chunk_ind:04d}.{ind:04d}.{inference_pass}.png",
                    )
                    inference_image_list[ind].save(save_path)
                # --- EVALUATION LOGIC ---
                if cfg.do_evaluation and inference_pass in [
                    "diffuse_albedo",
                    "normal",
                    "roughness"
                ]:
                    gt_image_pil = inference_image_list[ind]
                    input_image_filename = os.path.basename(
                        current_image_relative_path_list[ind]
                    )
                    gt_path = None
                    if inference_pass == "diffuse_albedo":
                        gt_path = os.path.join(
                            cfg.flare_albedo_dir, input_image_filename.zfill(8)
                        )
                    elif inference_pass == "normal":
                        gt_path = os.path.join(
                            cfg.flare_normal_dir, input_image_filename.zfill(8)
                        )
                    elif inference_pass == "roughness":
                        gt_path = os.path.join(
                            cfg.flare_roughness_dir, input_image_filename.zfill(8)
                        )
                    mask_path = os.path.join(
                            cfg.flare_mask_dir, input_image_filename
                        )
                    if gt_path and os.path.exists(gt_path):
                        gen_image_pil = Image.open(gt_path)
                        mask = plt.imread(mask_path)
                        psnr_val, ssim_val, lpips_val = calculate_metrics(
                            gen_pil=gen_image_pil,
                            gt_pil=gt_image_pil,
                            lpips_model=lpips_model,
                            lpips_transform=eval_transform_lpips,
                            device=device,
                            mask_np=mask,
                            model_dtype=cfg.torch_dtype,
                        )
                        metrics_data = {
                            "file": input_image_filename,
                            "pass": inference_pass,
                            "psnr": psnr_val,
                            "ssim": ssim_val,
                            "lpips": lpips_val,
                        }
                        all_metrics_data[inference_pass].append(metrics_data)
                    elif gt_path:
                        print(f"  [Eval] GT file not found, skipping: {gt_path}")
            if cfg.save_video:
                for ind in range(len(viz_images_uint8)):
                    viz_images_uint8[ind] = np.concatenate(
                        [
                            viz_images_uint8[ind],
                            np.asarray(inference_image_list[ind]),
                        ],
                        axis=1,
                    )

        if cfg.save_video:
            save_path = os.path.join(
                cfg.inference_save_dir,
                f"{video_relative_base_name}.{chunk_ind:04d}.viz.mp4",
            )
            imageio.mimsave(
                save_path, viz_images_uint8, fps=cfg.save_video_fps, codec="h264"
            )

Starting inference and evaluation...


Processing Videos: 0it [00:00, ?it/s]

  Processing Chunks:   0%|          | 0/1823 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [16]:
import pandas as pd

passes = ['diffuse_albedo', 'normal', 'roughness']

print("\n" + "="*30)
print(" FINAL EVALUATION METRICS ON 001")
print("="*30)
# In a single-node script, no 'gather' is needed.
all_metrics_list = all_metrics_data['diffuse_albedo'] + all_metrics_data['normal']  + all_metrics_data['roughness']
if len(all_metrics_list) == 0:
    print("No metrics gathered.")
else:
    df = pd.DataFrame(all_metrics_list)
    # Calculate and print averages
    for pass_name in passes:
        pass_df = df[df['pass'] == pass_name]
        if pass_df.empty:
            print(f"\nNo metrics calculated for {pass_name}.")
            continue
        avg_psnr = pass_df['psnr'].mean()
        avg_ssim = pass_df['ssim'].mean()
        avg_lpips = pass_df['lpips'].mean()
        print(f"\n--- Average Metrics for: {pass_name} ---")
        print(f"  PSNR:  {avg_psnr:.4f}")
        print(f"  SSIM:  {avg_ssim:.4f}")
        print(f"  LPIPS: {avg_lpips:.4f}")


 FINAL EVALUATION METRICS ON 001

--- Average Metrics for: diffuse_albedo ---
  PSNR:  21.9432
  SSIM:  0.7729
  LPIPS: 0.2166

--- Average Metrics for: normal ---
  PSNR:  14.3481
  SSIM:  0.6011
  LPIPS: 0.3045

--- Average Metrics for: roughness ---
  PSNR:  14.8098
  SSIM:  0.5179
  LPIPS: 0.4058


In [45]:
!realpath /tmp

/tmp


In [ ]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
import lpips
from PIL import Image
from tqdm import tqdm
import numpy as np
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root))
# Ensure we can import the specific metric function from your codebase
# (Assuming this script is placed where it can find 'material_aware_flare')
try:
    from material_aware_flare.eval import calculate_metrics
except ImportError:
    print("Warning: Could not import 'calculate_metrics'. Please ensure 'material_aware_flare' is in your python path.")

def evaluate_saved_intrinsics(
    prediction_dir,
    flare_albedo_dir,
    flare_normal_dir,
    flare_roughness_dir,
    flare_mask_dir,
    device='cuda'
):
    """
    Loads intrinsic channels saved by Diffusion Renderer and compares them 
    to Flare GT using the provided calculate_metrics function.
    """
    
    print(f"--- Starting Offline Evaluation ---")
    print(f"Prediction Directory: {prediction_dir}")
    
    # 1. Setup LPIPS and Transforms (Same as your main script)
    eval_transform_lpips = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    print(f"Loading LPIPS on {device}...")
    lpips_model = lpips.LPIPS(net='vgg').to(device)
    lpips_model.eval()

    # 2. Define file suffixes map to identify pass types
    # Based on code: f"{input_image_filename}_{inference_pass}.png"
    pass_suffixes = {
        'diffuse_albedo': '_diffuse_albedo.png',
        'normal': '_normal.png',
        'roughness': '_roughness.png'
    }
    
    gt_dirs = {
        'diffuse_albedo': flare_albedo_dir,
        'normal': flare_normal_dir,
        'roughness': flare_roughness_dir
    }

    all_metrics_data = []
    
    # 3. Get list of all files in the prediction directory
    if not os.path.exists(prediction_dir):
        print(f"Error: Prediction directory not found: {prediction_dir}")
        return

    pred_files = [f for f in os.listdir(prediction_dir) if f.endswith('.png')]
    print(f"Found {len(pred_files)} files in prediction directory.")

    # 4. Iterate and Evaluate
    for pred_filename in tqdm(pred_files, desc="Evaluating Images"):
        
        # Identify which pass this file belongs to
        current_pass = None
        original_filename = None
        
        for pass_name, suffix in pass_suffixes.items():
            if pred_filename.endswith(suffix):
                current_pass = pass_name
                # Extract original filename (e.g. '0001.png_normal.png' -> '0001.png')
                original_filename = pred_filename.replace(suffix, "")
                break
        
        # If file doesn't match known suffixes, skip
        if current_pass is None:
            continue

        # Define Paths
        pred_path = os.path.join(prediction_dir, pred_filename)
        
        # Logic for GT Filename: matches logic in your loop (zfill(8))
        # If original is "1812.png", zfill might not affect it depending on length, 
        # but we replicate your script's logic:
        gt_filename = original_filename.zfill(8) 
        
        gt_path = os.path.join(gt_dirs[current_pass], gt_filename)
        
        # If zfilled path doesn't exist, try the raw filename 
        # (in case original files are already correct length)
        if not os.path.exists(gt_path):
            gt_path = os.path.join(gt_dirs[current_pass], original_filename)

        mask_path = os.path.join(flare_mask_dir, original_filename)

        # Check existence
        if not os.path.exists(gt_path):
            # Try looking for jpg if png missing, or simple skip
            # print(f"GT missing for {original_filename} ({current_pass})")
            continue
            
        if not os.path.exists(mask_path):
             # print(f"Mask missing for {original_filename}")
             continue

        try:
            # Load Images
            gen_pil = Image.open(pred_path).convert("RGB")
            gt_pil = Image.open(gt_path).convert("RGB")
            
            # Resize GT to match Prediction (Pred is usually 512x512 based on your config)
            if gt_pil.size != gen_pil.size:
                gt_pil = gt_pil.resize(gen_pil.size, Image.BILINEAR)
            
            # Load Mask
            # Note: plt.imread scales 0-1 for png, 0-255 for jpg usually. 
            # calculate_metrics expects numpy mask.
            mask_np = plt.imread(mask_path)
            
            # Ensure mask is resized if necessary (simple resize via PIL then back to NP)
            if mask_np.shape[:2] != gen_pil.size[::-1]: # PIL is (W,H), NP is (H,W)
                mask_pil = Image.fromarray((mask_np * 255).astype(np.uint8) if mask_np.max() <= 1.0 else mask_np.astype(np.uint8))
                mask_pil = mask_pil.resize(gen_pil.size, Image.NEAREST)
                mask_np = np.array(mask_pil) / 255.0

            # Calculate Metrics
            psnr_val, ssim_val, lpips_val = calculate_metrics(
                gen_pil=gen_pil,
                gt_pil=gt_pil,
                lpips_model=lpips_model,
                lpips_transform=eval_transform_lpips,
                device=device,
                mask_np=mask_np,
                model_dtype=torch.float32 # Evaluating usually done in fp32
            )

            all_metrics_data.append({
                "file": original_filename,
                "pass": current_pass,
                "psnr": psnr_val,
                "ssim": ssim_val,
                "lpips": lpips_val,
            })

        except Exception as e:
            print(f"Error processing {pred_filename}: {e}")
            continue

    # 5. Aggregate and Print Results
    if not all_metrics_data:
        print("No metrics calculated. Check paths and filenames.")
        return

    df = pd.DataFrame(all_metrics_data)
    
    print("\n" + "="*30)
    print(" OFFLINE EVALUATION RESULTS")
    print("="*30)
    
    for pass_name in ['diffuse_albedo', 'normal', 'roughness']:
        pass_df = df[df['pass'] == pass_name]
        if pass_df.empty:
            print(f"\nNo data for {pass_name}")
            continue
            
        print(f"\n--- {pass_name} (n={len(pass_df)}) ---")
        print(f"  PSNR:  {pass_df['psnr'].mean():.4f}")
        print(f"  SSIM:  {pass_df['ssim'].mean():.4f}")
        print(f"  LPIPS: {pass_df['lpips'].mean():.4f}")
        
    # Option to save to CSV
    save_csv_path = os.path.join(prediction_dir, "offline_metrics.csv")
    df.to_csv(save_csv_path)
    print(f"\nDetailed metrics saved to: {save_csv_path}")


# Configuration extracted from your provided code
# NOTE: The script loop redefined 'video_relative_base_name' to an absolute path,
# so images are likely here:
saved_renderer_output = "/home/hleonhard/data/diff_renderer_001_MVI_1812"

# Flare GT Paths
gt_albedo = "/home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/albedo/"
gt_normal = "/home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/normal/"
gt_roughness = "/home/hleonhard/data/flare_models/001/MVI_1812/images_evaluation/roughness/"
gt_mask = '/home/hleonhard/data/flare_subject_data/001/MVI_1812/mask/'
evaluate_saved_intrinsics(
    prediction_dir=saved_renderer_output,
    flare_albedo_dir=gt_albedo,
    flare_normal_dir=gt_normal,
    flare_roughness_dir=gt_roughness,
    flare_mask_dir=gt_mask
)

--- Starting Offline Evaluation ---
Prediction Directory: /home/hleonhard/data/diff_renderer_001_MVI_1812
Loading LPIPS on cuda...
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/lpips/lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which w

Loading model from: /home/hleonhard/miniconda3/envs/diffusion_renderer/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth
Found 3648 files in prediction directory.


Evaluating Images: 100%|██████████| 3648/3648 [09:00<00:00,  6.75it/s]


 OFFLINE EVALUATION RESULTS

--- diffuse_albedo (n=1213) ---
  PSNR:  22.1366
  SSIM:  0.7862
  LPIPS: 0.2070

--- normal (n=1213) ---
  PSNR:  14.0095
  SSIM:  0.6035
  LPIPS: 0.3079

--- roughness (n=1213) ---
  PSNR:  15.7103
  SSIM:  0.5307
  LPIPS: 0.4202

Detailed metrics saved to: /home/hleonhard/data/diff_renderer_001_MVI_1812/offline_metrics.csv


In [5]:
import pandas as pd

def calculate_weighted_means():
    # --- Configuration ---
    n1 = 1213
    data1 = {
        'diffuse_albedo': {'psnr': 22.1366, 'ssim': 0.7862, 'lpips': 0.2070},
        'normal':         {'psnr': 14.0095, 'ssim': 0.6035, 'lpips': 0.3079},
        'roughness':      {'psnr': 15.7103, 'ssim': 0.5307, 'lpips': 0.4202}
    }

    n2 = 600
    data2 = {
        'diffuse_albedo': {'psnr': 21.9432, 'ssim': 0.7729, 'lpips': 0.2166},
        'normal':         {'psnr': 14.3481, 'ssim': 0.6011, 'lpips': 0.3045},
        'roughness':      {'psnr': 14.8098, 'ssim': 0.5179, 'lpips': 0.4058}
    }

    passes = ['diffuse_albedo', 'normal', 'roughness']
    metrics = ['psnr', 'ssim', 'lpips']
    
    total_n = n1 + n2
    aggregated_results = []

    print(f"Aggregating Set 1 (n={n1}) and Set 2 (n={n2})")
    print(f"Total samples: {total_n}")
    print("-" * 60)

    for p in passes:
        row = {'Pass': p}
        for m in metrics:
            val1 = data1[p][m]
            val2 = data2[p][m]
            
            # Weighted Average Formula
            weighted_avg = ((val1 * n1) + (val2 * n2)) / total_n
            row[m.upper()] = weighted_avg
            
        aggregated_results.append(row)

    df = pd.DataFrame(aggregated_results)
    
    df.set_index('Pass', inplace=True)
    pd.options.display.float_format = '{:.4f}'.format
    
    print(df)
    
    return df

calculate_weighted_means()

Aggregating Set 1 (n=1213) and Set 2 (n=600)
Total samples: 1813
------------------------------------------------------------
                  PSNR   SSIM  LPIPS
Pass                                
diffuse_albedo 22.0726 0.7818 0.2102
normal         14.1216 0.6027 0.3068
roughness      15.4123 0.5265 0.4154


,PSNR,SSIM,LPIPS
Pass,,,
diffuse_albedo,22.0726,0.7818,0.2102
normal,14.1216,0.6027,0.3068
roughness,15.4123,0.5265,0.4154
